In [1]:
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, str(Path.cwd().parent))
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List
from backtesting import Backtest

from strat.s_bb_breakout import (
    build_features,
    convert_to_ohlcv,
    BBBreakoutStrategy,
)
from core.enums import (
    g_open_col, g_high_col, g_low_col, g_close_col, g_volume_col, g_index_col,
)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

%matplotlib inline

Loading BokehJS ...

In [2]:
BUNDLE_PATH = Path('../data/bundle/test_etf_features_bundle.parquet')

df_bundle = pd.read_parquet(BUNDLE_PATH)
print(f"Bundle shape: {df_bundle.shape}")

def get_etf_symbols(df: pd.DataFrame) -> List[str]:
    symbols = set()
    for col in df.columns:
        if '_S_close_f32' in col:
            symbols.add(col.split('_')[0])
    return sorted(symbols)

etf_symbols = get_etf_symbols(df_bundle)
print(f"ETFs found: {etf_symbols}")

Bundle shape: (48107, 611)
ETFs found: ['GLD', 'QQQ', 'SPY', 'TLT', 'VWO']


In [3]:
def extract_etf_data(df_bundle: pd.DataFrame, symbol: str) -> pd.DataFrame:
    df = pd.DataFrame(index=df_bundle.index)
    df[g_open_col] = df_bundle[f'{symbol}_S_open_f32']
    df[g_high_col] = df_bundle[f'{symbol}_S_high_f32']
    df[g_low_col] = df_bundle[f'{symbol}_S_low_f32']
    df[g_close_col] = df_bundle[f'{symbol}_S_close_f32']
    df[g_volume_col] = df_bundle[f'{symbol}_S_volume_f64']
    return df.dropna()

def run_bb_backtest(df: pd.DataFrame, bb_period: int, bb_std: float, squeeze_lookback: int, atr_mult: float, rr: float, direction: str = "both") -> Dict:
    df = build_features(df, p_direction=direction, p_bb_period=bb_period, p_bb_std=bb_std, p_squeeze_lookback=squeeze_lookback)
    ohlcv_df = convert_to_ohlcv(df)
    if len(ohlcv_df) == 0:
        return None
    bt = Backtest(ohlcv_df, BBBreakoutStrategy, cash=100_000, commission=0.0002, trade_on_close=True, hedging=False, exclusive_orders=False)
    stats = bt.run(atr_mult=atr_mult, rr=rr, direction=direction)
    return {'stats': stats, 'return_pct': stats['Return [%]'], 'sharpe': stats['Sharpe Ratio'], 'max_dd': stats['Max. Drawdown [%]'], 'win_rate': stats['Win Rate [%]'], 'n_trades': stats['# Trades'], 'profit_factor': stats['Profit Factor']}

In [4]:
BB_PERIODS = [15, 20, 25,50]
BB_STDS = [1.5, 2.0, 2.5]
SQUEEZE_LOOKBACKS = [50, 100, 150]
ATR_MULTS = [1.5, 2.0, 2.5]
RR_RATIOS = [1.5, 2.0, 2.5]

total = len(BB_PERIODS) * len(BB_STDS) * len(SQUEEZE_LOOKBACKS) * len(ATR_MULTS) * len(RR_RATIOS)
print(f"Total combinations: {total}")

Total combinations: 324


In [5]:
etf_symbols[0:2]

['GLD', 'QQQ']

In [6]:
def optimize_etf(df_bundle, symbol, max_runs=100, direction="both"):
    df = extract_etf_data(df_bundle, symbol)
    print(f"{symbol}: {len(df)} rows")
    results = []
    count = 0
    for bb_period in BB_PERIODS:
        for bb_std in BB_STDS:
            for squeeze_lookback in SQUEEZE_LOOKBACKS:
                for atr_mult in ATR_MULTS:
                    for rr in RR_RATIOS:
                        count += 1
                        if count > max_runs:
                            break
                        try:
                            result = run_bb_backtest(df, bb_period, bb_std, squeeze_lookback, atr_mult, rr, direction)
                            if result:
                                results.append({'bb_period': bb_period, 'bb_std': bb_std, 'squeeze_lookback': squeeze_lookback, 'atr_mult': atr_mult, 'rr': rr, 'sharpe': result['sharpe'], 'return_pct': result['return_pct'], 'max_dd': result['max_dd'], 'win_rate': result['win_rate'], 'n_trades': result['n_trades'], 'profit_factor': result['profit_factor']})
                        except:
                            pass
    results_df = pd.DataFrame(results)
    if len(results_df) == 0:
        return None, None
    best_idx = results_df['sharpe'].idxmax()
    return results_df, results_df.loc[best_idx].to_dict()

In [7]:
MAX_RUNS = 324
DIRECTION = "both"

all_results = {}
all_best = {}

df_optim = df_bundle[:24000]

for symbol in etf_symbols[2:3]:
    print(f"\n{'='*60}\nOptimizing {symbol}...")
    results_df, best = optimize_etf(df_optim, symbol, max_runs=MAX_RUNS, direction=DIRECTION)
    if results_df is not None:
        all_results[symbol] = results_df
        all_best[symbol] = best
        print(f"Best Sharpe: {best['sharpe']:.4f}, Return: {best['return_pct']:.2f}%")


Optimizing SPY...
SPY: 24000 rows


Backtest.run:   0%|          | 0/23986 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/23986 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/23986 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/23986 [00:00<?, ?bar/s]

...

Backtest.run:   0%|          | 0/23986 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/23986 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/23986 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/23986 [00:00<?, ?bar/s]

KeyError: nan

In [ ]:
summary_data = []
for symbol, best in all_best.items():
    summary_data.append({'ETF': symbol, 'BB_period': int(best['bb_period']), 'BB_std': best['bb_std'], 'Squeeze': int(best['squeeze_lookback']), 'ATR_mult': best['atr_mult'], 'RR': best['rr'], 'Sharpe': round(best['sharpe'], 3) if np.isfinite(best['sharpe']) else 'N/A', 'Return %': round(best['return_pct'], 2), 'MaxDD %': round(best['max_dd'], 2), '# Trades': int(best['n_trades'])})

summary_df = pd.DataFrame(summary_data)
print("\n" + "="*80)
print("BB BREAKOUT OPTIMIZATION SUMMARY")
print("="*80)
print(summary_df.to_string(index=False))
summary_df

In [ ]:
def plot_heatmap(results_df, symbol):
    if results_df is None or len(results_df) == 0:
        return
    pivot = results_df.pivot_table(values='sharpe', index='bb_std', columns='bb_period', aggfunc='mean')
    plt.figure(figsize=(8, 6))
    sns.heatmap(pivot, annot=True, fmt='.2f', cmap='RdYlGn', center=0, cbar_kws={'label': 'Sharpe'})
    plt.title(f'{symbol} - BB Period vs Std')
    plt.tight_layout()
    plt.show()

for symbol in list(all_results.keys())[:2]:
    plot_heatmap(all_results[symbol], symbol)

In [ ]:
output_path = Path('results/bb_breakout_optimization.csv')
output_path.parent.mkdir(parents=True, exist_ok=True)
summary_df.to_csv(output_path, index=False)
print(f"Results saved to {output_path}")